In [ ]:
from src import *
from src.internals import *
from numpy import sin, cos
from dataclasses import field
from numpy import sqrt
from dataclasses import dataclass
import random
random.seed(42)

# Geometry defined by ray-object intersection
We already know how to define rays and cameras, but how do we define the geometry of objects in the scene? In ray tracing, geometry is typically defined by how rays intersect with objects. Each object has an intersection method that calculates if and where a ray intersects it, and returns information about the hit point, normal, and other relevant data for later shading. We will use mathematicly defined objects instead of mesh based objects in this notebook to better understand the underlying geometry and intersection calculations. We will start with a simple sphere, which is defined by its center and radius, and has a well-known analytical solution for ray-sphere intersection.

## Sphere definition
To define object we need to know how to calculate ray-sphere intersection. And normal at the hit point for shading. We will implement this in a custom sphere class that inherits from Primitive and implements the intersect method. The intersect method will take a ray and return a GeometryHit record if the ray intersects the sphere, or None if it misses. The normal_at method will calculate the normal vector at any point on the sphere's surface, which is needed for lighting calculations later on.


A sphere with centre $\mathbf{c}$ and radius $r$ satisfies:

$$\|\mathbf{x} - \mathbf{c}\|^2 = r^2$$

Substituting the parametric ray $\mathbf{r}(t) = \mathbf{o} + t\,\mathbf{d}$:

$$\|\mathbf{o} + t\,\mathbf{d} - \mathbf{c}\|^2 = r^2$$

Let $\mathbf{oc} = \mathbf{o} - \mathbf{c}$. Expanding gives the quadratic:

$$t^2 \underbrace{(\mathbf{d} \cdot \mathbf{d})}_{a}
 + 2t \underbrace{(\mathbf{oc} \cdot \mathbf{d})}_{b}
 + \underbrace{(\mathbf{oc} \cdot \mathbf{oc} - r^2)}_{c} = 0$$

The **discriminant** $\Delta = b^2 - 4ac$ tells us:

| $\Delta$ | Meaning |
|-----------|---------|
| $< 0$ | Ray misses the sphere entirely |
| $= 0$ | Ray is tangent (one intersection) |
| $> 0$ | Ray passes through (two intersections — take the smaller positive $t$) |

#### Limiting valid intersections with `t_min` and `t_max`
`t_min` and `t_max` define the interval of ray parameter $t$ in which an intersection is considered valid. This makes it possible to ignore hits behind the ray origin, avoid self-intersections caused by numerical error, and restrict the test to the relevant part of the ray.

In [ ]:
@dataclass
class MySphere(Primitive):
    # as default, we create a unit sphere that can be transformed later with transforms. This is a common practice in ray tracing to define objects in local space and then use transforms to position and scale them in the scene.
    center: Vertex = field(default_factory=lambda: Vertex(0, 0, 0))
    radius: float = 1.0

    def intersect(self, ray: Ray, t_min=0.001, t_max=float('inf')) -> GeometryHit | None:
        """
        Calculate intersection of ray with sphere and return hit record if intersection occurs.
        :param ray: Ray to test intersection with
        :param t_min: minimum valid distance for intersection
        :param t_max: maximum valid distance for intersection
        :return: Hit record if intersection occurs, else None
        """
        oc = ray.origin - self.center  # Vector from origin of ray to center of sphere

        # Quadratic coefficients
        a = ray.direction.dot(ray.direction)  # Usually = 1 if ray.direction is normalized
        b = 2.0 * oc.dot(ray.direction)  # Projection of oc onto the ray
        c = oc.dot(oc) - self.radius * self.radius  # Distance^2 from ray origin to sphere surface

        discriminant = b * b - 4 * a * c
        if discriminant < 0:
            return None  # no intersection - ray misses the sphere

        sqrt_disc = sqrt(discriminant)

        # Find the nearest root that lies in the acceptable range by calculating both roots using quadratic formula
        dist = (-b - sqrt_disc) / (2.0 * a)
        if dist < t_min or dist > t_max:
            dist = (-b + sqrt_disc) / (2.0 * a)
            if dist < t_min or dist > t_max:  # Point is out of range so no valid intersection
                return None

        # Calculate intersection in 3d space
        hit_point = ray.point_at(dist)


        outward = self.normal_at(hit_point)
        front_face = ray.direction.dot(outward) < 0.0
        normal = outward if front_face else -outward

        return GeometryHit(
            dist=dist,
            point=hit_point,
            normal=normal,
            front_face=front_face
        )


    def normal_at(self, point: Vertex) -> Vector:
        """
        Get the normal vector at a given point on the sphere's surface.
        :param point: Point on the sphere
        :return: Normal vector at that point
        """
        normal = (point - self.center) / self.radius
        return normal

## Simple color material

For now, we define a very simple color material that will be used to visualize the sphere in the scene.
This material returns the same constant color for every hit point on the sphere, which makes it useful for displaying geometry without any shading effects.

We implement it as a subclass of `Material` and override the `sample` method so that it always returns the chosen color.

More advanced materials and shading models will be introduced in later notebooks.

In [ ]:
@dataclass
class ColorMaterial(Material):
    color: Color = field(default_factory=lambda: Color.custom_rgb(255, 255, 255))
    def get_color(self) -> Color:
        return self.color

    def sample(self, hit: Vertex) -> Color:
        return self.color

## Visualize the sphere and its local geometry
Now that we have defined our sphere and a simple material, we can visualize it in the scene using the Visualizer. We will create a unit sphere centered at the origin and apply a color material to it. The visualizer will allow us to see the geometry of the sphere, its normals, and how rays intersect with it. This will help us understand the local geometry of the sphere and how it interacts with rays, which is fundamental for ray tracing and shading calculations later on.

> Unit sphere is a good starting point for understanding local geometry because it has a simple and well-known shape

In [ ]:
sphere = Object(
    geometry=Sphere(center=Vertex(0, 0, 0), radius=1.0),
    material=ColorMaterial(color=Color.custom_rgb(200, 200, 255))
)
vis = Visualizer()
vis.create_empty_scene(size=2.0, figsize=(12, 8), show_axes_labels=True, show_arrows=True, show_grid=True, show_axes=True, show_xyz_labels=True)
vis.visualize_objects([sphere])
vis.show_legend()
vis.set_title("Local Geometry - Unit Sphere", fontsize=12)
vis.savefig("unit_sphere.pdf", dpi=300)
vis.show()

## Visualize ray-sphere intersection
Now that we have our sphere defined and visualized, we can explore how rays intersect with it. We will set up a camera and generate rays from the camera through points on the image plane. We will then calculate the intersection of these rays with the sphere and visualize the hit points, normals, and the rays themselves. This will help us understand how the geometry of the sphere interacts with rays, which is crucial for ray tracing and shading calculations. We will also visualize the image plane point corresponding to the ray to see how it maps to the sphere's surface. This will provide a clear demonstration of ray-sphere intersection.

> **Try it yourself:** change the camera position, direction, and field of view to see how it affects the ray-sphere intersection. You can also experiment with different points on the image plane by changing the `u` and `v` coordinates to see how they correspond to different hit points on the sphere's surface. This is a great way to explore the relationship between the camera, rays, and geometry in ray tracing.

In [ ]:
camera = PinholeCamera(
    origin=Vertex(4, 2, 0),
    direction=Vector(-1, -0.5, 0),
    fov_deg=60,
)
vis.create_empty_scene(size=3.0, figsize=(12, 8), show_axes_labels=False, show_arrows=False, show_grid=True, show_axes=False, show_xyz_labels=False)
vis.visualize_camera_position_and_orientation(camera, show_frustum=True, frustum_depth=5, show_camera_orientation=False, show_plane=True)
vis.visualize_objects([sphere])

u = -0.1
v = -0.15

vis.visualize_image_plane_point(camera, u, v, label=True)

ray = camera.make_ray(u, v)
hit = sphere.intersect(ray)

vis.visualize_ray(ray, length=5.0, color='magenta', opacity=0.3, ended_by_hit_point=hit)
vis.visualize_normal_at_hit_point(hit,length=1.2)
vis.visualize_closest_intersection(ray, objects=[sphere], intersection_opacity=0.9)


vis.savefig("camera_ray.pdf", dpi=300, fontsize=13)
vis.show()

## Visualising normals as colour

Before we have any lighting, we can already verify our normals are correct by mapping
each component to an RGB channel:

$$\text{colour} = \frac{\mathbf{n} + 1}{2} = \left(\frac{n_x+1}{2},\; \frac{n_y+1}{2},\; \frac{n_z+1}{2}\right)$$

This maps $[-1, +1]$ to $[0, 1]$. A normal pointing right $(+X)$ appears red, up $(+Y)$
green, toward the camera $(+Z)$ blue. If the sphere looks like a smooth colour gradient
across its surface, the normals are correct.

#### Visualize rays and their intersections with the sphere

In [ ]:
vis.create_empty_scene(size=4.0, figsize=(12, 8), show_axes_labels=False, show_arrows=False, show_grid=True, show_axes=True, show_xyz_labels=True)
vis.visualize_camera_position_and_orientation(camera, show_frustum=True, frustum_depth=5, show_camera_orientation=False)
vis.visualize_objects([sphere])

# create random rays and visualize their intersections with the sphere (if they intersect)
for i in range(500):
    u = random.uniform(-1.0, 1.0)
    v = random.uniform(-1.0, 1.0)

    ray = camera.make_ray(u, v)

    hit_point : SurfaceInteraction= sphere.intersect(ray)

    vis.visualize_ray(ray, length=5.0, color='magenta', opacity=0.01, ended_by_hit_point=hit_point)
    vis.visualize_normal_at_hit_point(hit_point)
    vis.visualize_closest_intersection(ray, objects=[sphere], intersection_opacity=0.8)

vis.set_title("Intersection Points Visualized with Normals", fontsize=13)
vis.savefig("camera_ray.png", dpi=300)
vis.show()

#### Let's render the sphere with a shader
- In next notebooks we will learn how to implement different shading models like this.

In [ ]:
my_sphere = Object(
    geometry=MySphere(),
    material=ColorMaterial(color=Color.custom_rgb(100, 100, 255))
)

rt = LinearRenderLoop(
    scene=Scene(
        camera=camera,
        objects=[my_sphere]
    ),
    render_config=RenderConfig(resolution=Resolution.R360p, samples_per_pixel=2, max_depth=1),
    preview_config=PreviewConfig(progress_display=ProgressDisplay.TQDM_IMAGE_PREVIEW),
    shading_model=NormalShader()
)
rendered_image = rt.render("images/RGB_normal_map.png")
ipynb_display_images(rendered_image)

## Plane definition
- A plane is an unbounded primitive, unlike a sphere, so it is less suitable for efficient ray tracing because it cannot be enclosed in a finite bounding box for acceleration structures. However, it is very simple to define and is useful for educational purposes, since it clearly demonstrates ray-plane intersection and highlights how it differs from ray-sphere intersection.

A plane is defined by a point $\mathbf{p}$ on its surface and a normal vector $\mathbf{n}$.
A point $\mathbf{x}$ lies on the plane if

$$
(\mathbf{x} - \mathbf{p}) \cdot \mathbf{n} = 0 .
$$

For a ray

$$
\mathbf{r}(t) = \mathbf{o} + t\,\mathbf{d},
$$

the intersection parameter is obtained by substitution into the plane equation:

$$
t = \frac{(\mathbf{p} - \mathbf{o}) \cdot \mathbf{n}}{\mathbf{d} \cdot \mathbf{n}} .
$$




> **Try it yourself:** implement your own object that follows the `Primitive` interface and defines a different geometry, such as a cylinder, cone, or torus.

In [ ]:
@dataclass
class MyPlane(Primitive):
    """
    Plane in 3D space defined by a point, normal, and color.
    """
    point: Vertex = field(default_factory=lambda: Vertex(0, 0, 0))  # a point on the plane
    normal: Vector = field(default_factory=lambda: Vector(0, 1, 0))  # normal vector to the plane

    def intersect(self, ray: Ray, t_min=0.001, t_max=float('inf')) -> GeometryHit | None:
        """
        Calculate intersection of ray with plane.
        :param ray: Ray to test intersection with
        :param t_min: minimum valid distance for intersection
        :param t_max: maximum valid distance for intersection
        :return: Hit record if intersection occurs, else None
        """
        denominator = ray.direction.dot(self.normal)
        if abs(denominator) < 1e-6:
            return None  # Ray is parallel to the plane
        dist = (self.point - ray.origin).dot(self.normal) / denominator

        # out of bounds check
        if dist < t_min or dist > t_max:
            return None

        hit_point = ray.point_at(dist)
        normal = self.normal_at(hit_point)

        # check if the ray is hitting the front face or back face of the plane and flip the normal if necessary for correct lighting calculations
        if ray.direction.dot(normal) > 0.0:
            normal = -normal

        return GeometryHit(
            dist=dist,
            point=hit_point,
            normal=normal,
            front_face=ray.direction.dot(normal) < 0.0,
        )

    def normal_at(self, point: Vertex) -> Vector:
        """
        Get the normal vector at a given point on the plane's surface.
        :param point: Point on the plane
        :return: Normal vector at that point
        """
        return self.normal

#### Let's visualize the plane together with the sphere
> **Try it yourself:** change the plane's point and normal to see how it affects rendering with normal mapping shader.

In [ ]:
objects = [
    my_sphere,
    Object(
        geometry=MyPlane(point=Vertex(0, -1, 0), normal=Vector(0, 1, 0)),
        material=ColorMaterial(color=Color.custom_rgb(200, 255, 200))
    )
]

rt = LinearRenderLoop(
    scene=Scene(
        camera=camera,
        objects=objects
    ),
    render_config=RenderConfig(resolution=Resolution.R360p, samples_per_pixel=2, max_depth=1),
    preview_config=PreviewConfig(progress_display=ProgressDisplay.TQDM_IMAGE_PREVIEW),
    shading_model=NormalShader()
)


rendered_image = rt.render("images/RGB_normal_map_2.png")
ipynb_display_images(rendered_image)

---
# Object Transformations

So far every object has been defined directly in world space — a sphere at a specific
centre, a box between two corners. A more flexible approach is to define geometry in a
canonical **local coordinate system** and then place it in the world using a
**transformation matrix**.

A transformation is expressed as a single $4 \times 4$ matrix $\mathbf{M}$ built from
three elementary operations that can be combined in any order but are conventionally applied in the order of **scaling**, then **rotation**, then **translation** because we want to scale and rotate the object around its local origin before moving it to its final position in world space. The order matters.
Scaling then translating gives a different result than translating then scaling.

$$\mathbf{M} = \mathbf{T} \cdot \mathbf{R} \cdot \mathbf{S}$$

| Operation | Effect |
|---|---|
| $\mathbf{T}$ — translation | moves the object to a position in world space |
| $\mathbf{R}$ — rotation | rotates around the $X$, $Y$, or $Z$ axis |
| $\mathbf{S}$ — scaling | stretches or shrinks along each axis independently |

### Transforming the Ray, Not the Geometry

Rather than transforming every vertex of the geometry, the ray tracer does the
opposite: it transforms the **ray** into the object's local space using the
**inverse** of $\mathbf{M}$, performs the intersection there in the simple canonical
form, then transforms the resulting hit point and normal **back** to world space.

The hit point transforms with $\mathbf{M}$ directly, but the normal requires the
**inverse transpose** $(\mathbf{M}^{-1})^\top$ to remain perpendicular to the
surface after non-uniform scaling:

$$\mathbf{p}_{world} = \mathbf{M}\,\mathbf{p}_{local}, \qquad
\mathbf{n}_{world} = (\mathbf{M}^{-1})^\top\,\mathbf{n}_{local}$$

The `Object` class handles all of this automatically — you only need to call
`.translate()`, `.rotate_y()`, `.scale()`

In order from left to right: original unit sphere, scaled, translated, rotated.

> **Note:** Do not mix these approaches. Choose either global coordinates or transformations on unit objects.

In [ ]:
sphere_local = Object(
    geometry=Sphere(),
    material=ColorMaterial(color=Color.custom_rgb(200, 200, 255))
)

sphere_translated = Object(
    geometry=Sphere(),
    material=ColorMaterial(color=Color.custom_rgb(255, 200, 200))
).translate(2, 1, -1.3)

sphere_scaled = Object(
    geometry=Sphere(),
    material=ColorMaterial(color=Color.custom_rgb(200, 255, 200))
).scale(1.7, 0.1, 1.7)   # flatten into an ellipsoid

sphere_rotated = Object(
    geometry=Sphere(),
    material=ColorMaterial(color=Color.custom_rgb(0, 0, 0))
).scale(1.5, 3.0, 1.2).rotate_x(45.0).translate(-5, 1, 0)

objects = [sphere_local, sphere_scaled, sphere_translated, sphere_rotated]

#### Visualize the transformed objects

In [ ]:
vis = Visualizer()
vis.create_empty_scene(size=5.0, figsize=(12, 8), show_grid=True, show_xyz_labels=True)
vis.visualize_objects(objects)

#vis.visualize_camera_position_and_orientation(camera, show_frustum=True, frustum_depth=9, show_camera_orientation=False) # uncomment to see camera position and frustum in the scene
vis.savefig(filename="transformations.pdf")
vis.show()

#### Render the transformed objects with a shader that visualizes normals as colour

In [ ]:
renderer = LinearRenderLoop(
    scene=Scene(
        camera=camera,
        objects=objects
    ),
    render_config=RenderConfig(resolution=Resolution.R360p, samples_per_pixel=1, max_depth=1),
    preview_config=PreviewConfig(progress_display=ProgressDisplay.TQDM_IMAGE_PREVIEW),
    shading_model=NormalShader()
)
img = renderer.render("images/transformations.png")
ipynb_display_images(img)

---
# BONUS

This demonstrates how interesting defining geometry with ray intersections can be, and how it opens up possibilities for more complex shapes that don't have simple analytical solutions.
> **Try it yourself:** try to implement your own hit function and define object by it.
## Implicit surfaces defined by signed distance functions

A particularly powerful type of implicit function is the **signed distance function**, which returns the shortest distance from any point $\mathbf{p}$ to the surface, with the sign indicating whether $\mathbf{p}$ is inside or outside:

$$f(\mathbf{p}) \begin{cases} > 0 & \text{outside} \\\ = 0 & \text{on the surface} \\\ < 0 & \text{inside} \end{cases}$$

For example, a sphere of radius $r$ centred at the origin has the SDF $f(\mathbf{p}) = \|\mathbf{p}\| - r$. This means that points outside the sphere will have a positive distance, points on the surface will have zero distance, and points inside the sphere will have a negative distance. This method is called **ray marching** and is a powerful technique for rendering complex implicit surfaces.


In [ ]:
@dataclass
class ImplicitSurface(Primitive):
    """
    Represents a surface defined by an implicit function, specifically a signed distance function.
    """

    sdf: callable  # signed distance function

    def intersect(self, ray: Ray, t_min: float = 1e-3, t_max: float = float('inf')) -> GeometryHit | None:

        # how many steps to take along the ray when marching towards the surface, and how close we need to be to consider it a hit.
        steps = 100
        e_3 = 1e-3

        t = t_min
        prev_dist = self.sdf(ray.point_at(t))

        for _ in range(steps):
            point = ray.point_at(t)
            dist = self.sdf(point)

            # hit if we're close enough to the surface
            if abs(dist) < e_3:
                normal = self.normal_at(point)
                front_face = ray.direction.dot(normal) < 0.0
                return GeometryHit(dist=t, point=point, normal=normal, front_face=front_face)

            # detect sign change
            if prev_dist * dist < 0:
                p_lo, p_hi = t - abs(prev_dist), t

                # binary search to refine the hit point between p_lo and p_hi
                for _ in range(18):
                    t_mid = (p_lo + p_hi) * 0.5
                    d_mid = self.sdf(ray.point_at(t_mid))
                    # close enough to the surface (hit)
                    if abs(d_mid) < e_3:
                        break
                    # half the interval based on the sign of the distance at the midpoint
                    if prev_dist * d_mid < 0:
                        p_hi = t_mid
                    else:
                        p_lo = t_mid

                # after binary search, we take the midpoint of hit interval as the final hit point for better accuracy
                t_hit = (p_lo + p_hi) * 0.5
                point = ray.point_at(t_hit)
                normal = self.normal_at(point)
                front_face = ray.direction.dot(normal) < 0.0
                return GeometryHit(dist=t_hit, point=point, normal=normal, front_face=front_face)

            prev_dist = dist
            t += abs(dist)

            if t >= t_max:
                break

        return None


    def normal_at(self, point: Vertex) -> Vertex:
        """
        Estimate normal at a point on the implicit surface using central differences. Like numerical gradient of the SDF.
        :param point: Point on the surface
        :return: Normal vector at that point
        """
        e_5 = 1e-5
        dx = self.sdf(Vertex(point.x + e_5, point.y, point.z)) - self.sdf(Vertex(point.x - e_5, point.y, point.z))
        dy = self.sdf(Vertex(point.x, point.y + e_5, point.z)) - self.sdf(Vertex(point.x, point.y - e_5, point.z))
        dz = self.sdf(Vertex(point.x, point.y, point.z + e_5)) - self.sdf(Vertex(point.x, point.y, point.z - e_5))
        normal = Vector(dx, dy, dz).normalize()
        return normal

#### Sphere defined by an implicit surface
- This is the same sphere as before, but now defined by an implicit surface using a signed distance function.

In [ ]:
my_implicit_object = Object(
    geometry=ImplicitSurface(sdf=lambda p: sqrt(p.x**2 + p.y**2 + p.z**2) - 1), # sphere of radius 1 centered at origin
    material=ColorMaterial(color=Color.custom_rgb(255, 200, 200))
)

renderer = LinearRenderLoop(
    scene=Scene(
        camera=camera,
        objects=[my_implicit_object]
    ),
    render_config=RenderConfig(resolution=Resolution.R360p, samples_per_pixel=1, max_depth=1),
    preview_config=PreviewConfig(progress_display=ProgressDisplay.TQDM_IMAGE_PREVIEW),
    shading_model=NormalShader()
)
img = renderer.render("images/RGB_implicit_surface.png")
ipynb_display_images(img)

#### More complex implicit surface defined by a wavy function
- This surface is defined by a more complex implicit function that creates a wavy pattern.
> **Try it yourself:** experiment with different implicit functions to create interesting shapes and patterns.

In [ ]:
camera = PinholeCamera(
    origin=Vertex(4, 4, 0),
    direction=Vector(-1, -0.65, 0),
    fov_deg=70,
)

myImplicitSurface = Object(
    geometry=ImplicitSurface(lambda p: abs(p.y - 0.6 * sin(2 * p.x) * cos(3 * p.z)) - 0.9), # wavy
    material=ColorMaterial(color=Color.custom_rgb(200, 255, 200))
)
scene=Scene(
    camera=camera,
    objects=[myImplicitSurface]
)

scene.set_skybox("white")
renderer = LinearRenderLoop(
    scene=scene,
    render_config=RenderConfig(resolution=Resolution.R360p, samples_per_pixel=1, max_depth=1),
    preview_config=PreviewConfig(progress_display=ProgressDisplay.TQDM_IMAGE_PREVIEW),
    shading_model=NormalShader()
)
rendered_image = renderer.render("images/implicit_surface.png")
ipynb_display_images(rendered_image)

---
# Summary

In this notebook, we introduced the fundamental geometric primitives and their intersection logic with rays, this is one of the most used operations in ray tracing.

- **Sphere definition** — we defined a `Sphere` class with an `intersect()` method that solves the quadratic equation to find ray-sphere intersections and a `normal_at()` method to compute surface normals.
- **Object transformations** — we explored how to use transformation matrices to position, rotate, and scale objects in the scene, and how the ray tracer handles transforming rays into local space for intersection tests.
- **Implicit surfaces** — we introduced the concept of implicit surfaces defined by signed distance functions and implemented a `ImplicitSurface` class that uses sphere tracing to find intersections without needing explicit geometry.
- **Visualization** — we visualized rays, hit points, and normals to verify our intersection logic and rendered the sphere with a normal shader to confirm the correctness of our calculations and created simple flat colour for verification of hit points.